In [2]:
from __future__ import annotations

import json
import re
import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated, Any

# --- Pydantic V1 for LangChain Compatibility ---
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# --- LangChain & Ollama ---
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from datetime import date



In [ ]:
# ==========================================
# 0. HELPER: Clean JSON from Markdown
# ==========================================
def clean_json_output(text: str) -> str:
    """Removes markdown code fences to ensure json.loads can parse the output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

# ==========================================
# 1. ROBUST SCHEMAS (Pydantic V1)
# ==========================================

class Task(BaseModel):
    id: int = Field(default=0)
    title: str = Field(default="Section")

    @root_validator(pre=True)
    def map_deepseek_task_keys(cls, values: dict[str, Any]) -> dict[str, Any]:
        mapping = {
            "section_title": "title", "section_goal": "goal",
            "content": "bullets", "target_word_count": "target_words"
        }
        for old_key, new_key in mapping.items():
            if old_key in values:
                values[new_key] = values.pop(old_key)
        return values

    goal: str = Field(..., description="Section goal")
    bullets: List[str] = Field(..., min_length=1)
    target_words: int = Field(..., description="Word count")
    
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False

class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task]

    @root_validator(pre=True)
    def fix_deepseek_plan(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "blog_plan" in values: return values["blog_plan"]
        if "plan" in values and isinstance(values["plan"], list):
            return {
                "blog_title": "DeepSeek Generated Blog",
                "audience": "Developers",
                "tone": "Technical",
                "tasks": values["plan"]
            }
        return values

class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

    @root_validator(pre=True)
    def fix_deepseek_router(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "router_decision" in values:
            values = values["router_decision"]
            
        # 1. Default mode if missing
        if "mode" not in values:
            values["mode"] = "hybrid" if values.get("needs_research") else "closed_book"

        # 2. FORCE needs_research=True if mode is hybrid/open_book
        # This fixes the issue where it skips research despite being in hybrid mode
        if values.get("mode") in ["hybrid", "open_book"]:
            values["needs_research"] = True
            
            # Ensure we have at least one query if none provided
            if not values.get("queries"):
                values["queries"] = ["latest trends and best practices"]

        return values
    
class ImageSpec(BaseModel):
    placeholder: str = Field(..., description="e.g. [[IMAGE_1]]")
    filename: str = Field(..., description="Save under images/, e.g. qkv_flow.png")
    alt: str
    caption: str
    prompt: str = Field(..., description="Prompt to send to the image model.")
    size: Literal["1024x1024", "1024x1536", "1536x1024"] = "1024x1024"
    quality: Literal["low", "medium", "high"] = "medium"


class GlobalImagePlan(BaseModel):
    md_with_placeholders: str
    images: List[ImageSpec] = Field(default_factory=list)



class State(TypedDict):
    topic: str

    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)

    # reducer/image
    merged_md: str
    md_with_placeholders: str
    image_specs: List[dict]

    final: str

# ==========================================
# 2. LLM SETUP
# ==========================================
llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud", 
    temperature=0,
)

# ==========================================
# 3. ROUTER NODE
# ==========================================
ROUTER_SYSTEM = """You are a routing module. Decide if web research is needed.
Return STRICT JSON (no markdown):
{
  "needs_research": boolean,
  "mode": "hybrid", 
  "queries": ["query1", "query2"]
}
Modes:
- closed_book: Concepts only.
- hybrid: Concepts + Examples (Requires Research).
- open_book: News/Trends (Requires Research).
"""

def router_node(state: State) -> dict:
    print(f"--- Router Node (Topic: {state['topic']}) ---")
    response = llm.invoke(
        [
            SystemMessage(content=ROUTER_SYSTEM),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        # The validator logic in RouterDecision will now FORCE research=True for hybrid
        decision = RouterDecision(**data)
        
        print(f"Mode: {decision.mode} | Research Required: {decision.needs_research}")
        return {
            "needs_research": decision.needs_research,
            "mode": decision.mode,
            "queries": decision.queries,
        }
    except Exception as e:
        print(f"Router Error: {e}. Defaulting to Hybrid Research.")
        return {
            "needs_research": True, 
            "mode": "hybrid", 
            "queries": [f"{state['topic']} trends 2025", f"{state['topic']} best practices"]
        }

def route_next(state: State) -> str:
    # This will now correctly route to 'research' because needs_research is enforced
    return "research" if state["needs_research"] else "orchestrator"

# ==========================================
# 4. RESEARCH NODE
# ==========================================
def _tavily_search(query: str, max_results: int = 3) -> List[dict]:
    print(f"  > Searching Tavily: {query}")
    try:
        tool = TavilySearchResults(max_results=max_results)
        results = tool.invoke({"query": query})
        normalized = []
        for r in results or []:
            normalized.append({
                "title": r.get("title", "No Title"),
                "url": r.get("url", ""),
                "snippet": r.get("content", "") or r.get("snippet", ""),
                "published_at": r.get("published_date")
            })
        return normalized
    except Exception as e:
        print(f"    Tavily API Error: {e}")
        return []

RESEARCH_SYSTEM = """Synthesize search results into a JSON object.
Return STRICT JSON (no markdown):
{
  "evidence": [
    { "title": "...", "url": "...", "snippet": "..." }
  ]
}
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", [])[:3]
    if not queries:
        queries = [f"{state['topic']} analysis"]

    raw_results = []
    for q in queries:
        raw_results.extend(_tavily_search(q))

    if not raw_results:
        print("  > No results found.")
        return {"evidence": []}

    print("  > Synthesizing evidence...")
    response = llm.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw Results:\n{str(raw_results)[:10000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        items = data if isinstance(data, list) else data.get("evidence", [])
        
        valid_evidence = []
        seen_urls = set()
        for item in items:
            if item.get("url") and item["url"] not in seen_urls:
                valid_evidence.append(item)
                seen_urls.add(item["url"])
                
        print(f"  > Found {len(valid_evidence)} valid evidence items.")
        return {"evidence": valid_evidence}

    except Exception as e:
        print(f"  > Research Parse Error: {e}")
        return {"evidence": []}

# ==========================================
# 5. ORCHESTRATOR NODE
# ==========================================
ORCH_SYSTEM = """Create a blog plan.
Return STRICT JSON (no markdown):
{
  "blog_title": "...",
  "audience": "...",
  "tone": "...",
  "tasks": [
    { "title": "...", "goal": "...", "bullets": ["..."], "target_words": 200 }
  ]
}
"""

def orchestrator_node(state: State) -> dict:
    print("--- Orchestrator Node ---")
    evidence = state.get("evidence", [])
    
    response = llm.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Evidence: {[e.get('title') for e in evidence[:5]]}"
                )
            ),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        plan = Plan(**data)
        return {"plan": plan.dict()}
    except Exception as e:
        print(f"Plan Parse Error: {e}")
        # Fallback Plan
        fallback_plan = Plan(
            blog_title=f"Guide to {state['topic']}",
            audience="Developers",
            tone="Technical",
            tasks=[
                Task(id=1, title="Overview", goal="Intro", bullets=["Key concept 1", "Key concept 2"], target_words=200)
            ]
        )
        return {"plan": fallback_plan.dict()}

# ==========================================
# 6. FANOUT & WORKER
# ==========================================
def fanout(state: State):
    tasks_with_ids = []
    if state["plan"]:
        plan_data = state["plan"]
        tasks = plan_data.get("tasks", [])
        
        for i, task_data in enumerate(tasks):
            if "id" not in task_data or task_data["id"] == 0:
                task_data["id"] = i + 1
            tasks_with_ids.append(task_data)

    return [
        Send("worker", {
            "task": task,
            "topic": state["topic"],
            "plan": state["plan"],
            "evidence": state.get("evidence", []),
        }) for task in tasks_with_ids
    ]

WORKER_SYSTEM = """You are a technical writer. Write ONE section in Markdown.
Constraints:
- Use 

[Image of X]
 tags to suggest relevant diagrams.
- Cover all bullets.
- Use evidence URLs for citations if available.
- Start with '## Title'.
- Do NOT output JSON. Output raw Markdown.
"""

def worker_node(payload: dict) -> dict:
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = payload.get("evidence", [])
    
    bullets_text = "\n- " + "\n- ".join(task.bullets)
    evidence_text = "\n".join([f"- {e.get('title')} ({e.get('url')})" for e in evidence[:10]])

    print(f"Writing Section: {task.title}")
    
    response = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Section: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Bullets:{bullets_text}\n"
                    f"Evidence:\n{evidence_text}\n"
                )
            ),
        ]
    )
    return {"sections": [(task.id, response.content.strip())]}

# ============================================================
# 7) ReducerWithImages (subgraph)
#    merge_content -> decide_images -> generate_and_place_images
# ============================================================
def merge_content(state: State) -> dict:

    plan = state["plan"]

    ordered_sections = [md for _, md in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered_sections).strip()
    merged_md = f"# {plan.blog_title}\n\n{body}\n"
    return {"merged_md": merged_md}


DECIDE_IMAGES_SYSTEM = """You are an expert technical editor.
Decide if images/diagrams are needed for THIS blog.

Rules:
- Max 3 images total.
- Each image must materially improve understanding (diagram/flow/table-like visual).
- Insert placeholders exactly: [[IMAGE_1]], [[IMAGE_2]], [[IMAGE_3]].
- If no images needed: md_with_placeholders must equal input and images=[].
- Avoid decorative images; prefer technical diagrams with short labels.
Return strictly GlobalImagePlan.
"""

def decide_images(state: State) -> dict:
    
    planner = llm.with_structured_output(GlobalImagePlan)
    merged_md = state["merged_md"]
    plan = state["plan"]
    assert plan is not None

    image_plan = planner.invoke(
        [
            SystemMessage(content=DECIDE_IMAGES_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Topic: {state['topic']}\n\n"
                    "Insert placeholders + propose image prompts.\n\n"
                    f"{merged_md}"
                )
            ),
        ]
    )

    return {
        "md_with_placeholders": image_plan.md_with_placeholders,
        "image_specs": [img.model_dump() for img in image_plan.images],
    }


def _gemini_generate_image_bytes(prompt: str) -> bytes:
    """
    Returns raw image bytes generated by Gemini.
    Requires: pip install google-genai
    Env var: GOOGLE_API_KEY
    """
    from google import genai
    from google.genai import types

    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise RuntimeError("GOOGLE_API_KEY is not set.")

    client = genai.Client(api_key=api_key)

    resp = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"],
            safety_settings=[
                types.SafetySetting(
                    category="HARM_CATEGORY_DANGEROUS_CONTENT",
                    threshold="BLOCK_ONLY_HIGH",
                )
            ],
        ),
    )

    # Depending on SDK version, parts may hang off resp.candidates[0].content.parts
    parts = getattr(resp, "parts", None)
    if not parts and getattr(resp, "candidates", None):
        try:
            parts = resp.candidates[0].content.parts
        except Exception:
            parts = None

    if not parts:
        raise RuntimeError("No image content returned (safety/quota/SDK change).")

    for part in parts:
        inline = getattr(part, "inline_data", None)
        if inline and getattr(inline, "data", None):
            return inline.data

    raise RuntimeError("No inline image bytes found in response.")


def generate_and_place_images(state: State) -> dict:

    plan = state["plan"]
    assert plan is not None

    md = state.get("md_with_placeholders") or state["merged_md"]
    image_specs = state.get("image_specs", []) or []

    # If no images requested, just write merged markdown
    if not image_specs:
        filename = f"{plan.blog_title}.md"
        Path(filename).write_text(md, encoding="utf-8")
        return {"final": md}

    images_dir = Path("images")
    images_dir.mkdir(exist_ok=True)

    for spec in image_specs:
        placeholder = spec["placeholder"]
        filename = spec["filename"]
        out_path = images_dir / filename

        # generate only if needed
        if not out_path.exists():
            try:
                img_bytes = _gemini_generate_image_bytes(spec["prompt"])
                out_path.write_bytes(img_bytes)
            except Exception as e:
                # graceful fallback: keep doc usable
                prompt_block = (
                    f"> **[IMAGE GENERATION FAILED]** {spec.get('caption','')}\n>\n"
                    f"> **Alt:** {spec.get('alt','')}\n>\n"
                    f"> **Prompt:** {spec.get('prompt','')}\n>\n"
                    f"> **Error:** {e}\n"
                )
                md = md.replace(placeholder, prompt_block)
                continue

        img_md = f"![{spec['alt']}](images/{filename})\n*{spec['caption']}*"
        md = md.replace(placeholder, img_md)

    filename = f"{plan.blog_title}.md"
    Path(filename).write_text(md, encoding="utf-8")
    return {"final": md}

# build reducer subgraph
reducer_graph = StateGraph(State)
reducer_graph.add_node("merge_content", merge_content)
reducer_graph.add_node("decide_images", decide_images)
reducer_graph.add_node("generate_and_place_images", generate_and_place_images)
reducer_graph.add_edge(START, "merge_content")
reducer_graph.add_edge("merge_content", "decide_images")
reducer_graph.add_edge("decide_images", "generate_and_place_images")
reducer_graph.add_edge("generate_and_place_images", END)
reducer_subgraph = reducer_graph.compile()

reducer_subgraph



# -----------------------------
# 8) Build main graph
# -----------------------------
g = StateGraph(State)
g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_subgraph)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")

g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()
app


# -----------------------------
# 10) Runner
# -----------------------------
def run(topic: str, as_of: Optional[str] = None):
    if as_of is None:
        as_of = date.today().isoformat()

    out = app.invoke(
        {
            "topic": topic,
            "mode": "",
            "needs_research": False,
            "queries": [],
            "evidence": [],
            "plan": None,
            "as_of": as_of,
            "recency_days": 7,
            "sections": [],
            "merged_md": "",
            "md_with_placeholders": "",
            "image_specs": [],
            "final": "",
        }
    )

    return out

In [8]:
from __future__ import annotations

import json
import re
import os
import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated, Any

# --- Pydantic V1 for LangChain Compatibility ---
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# --- LangChain & Ollama ---
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

# ==========================================
# 0. HELPER: Clean JSON from Markdown
# ==========================================
def clean_json_output(text: str) -> str:
    """Removes markdown code fences to ensure json.loads can parse the output."""
    text = text.strip()
    # Remove ```json or ``` at start/end
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

# ==========================================
# 1. ROBUST SCHEMAS (Pydantic V1)
# ==========================================

class Task(BaseModel):
    id: int = Field(default=0)
    title: str = Field(default="Section")

    @root_validator(pre=True)
    def map_deepseek_task_keys(cls, values: dict[str, Any]) -> dict[str, Any]:
        mapping = {
            "section_title": "title", "section_goal": "goal",
            "content": "bullets", "target_word_count": "target_words"
        }
        for old_key, new_key in mapping.items():
            if old_key in values:
                values[new_key] = values.pop(old_key)
        return values

    goal: str = Field(..., description="Section goal")
    bullets: List[str] = Field(..., min_length=1)
    target_words: int = Field(..., description="Word count")
    
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False

class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task]

    @root_validator(pre=True)
    def fix_deepseek_plan(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "blog_plan" in values: return values["blog_plan"]
        if "plan" in values and isinstance(values["plan"], list):
            return {
                "blog_title": "DeepSeek Generated Blog",
                "audience": "Developers",
                "tone": "Technical",
                "tasks": values["plan"]
            }
        return values

class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None
    snippet: Optional[str] = None

class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

    @root_validator(pre=True)
    def fix_deepseek_router(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "router_decision" in values:
            values = values["router_decision"]
            
        # 1. Default mode if missing
        if "mode" not in values:
            values["mode"] = "hybrid" if values.get("needs_research") else "closed_book"

        # 2. FORCE needs_research=True if mode is hybrid/open_book
        if values.get("mode") in ["hybrid", "open_book"]:
            values["needs_research"] = True
            if not values.get("queries"):
                values["queries"] = ["latest trends and best practices"]

        return values

# --- Image Generation Schemas ---
class ImageSpec(BaseModel):
    placeholder: str = Field(..., description="e.g. [[IMAGE_1]]")
    filename: str = Field(..., description="e.g. architecture_diagram.png")
    alt: str
    caption: str
    prompt: str = Field(..., description="Detailed prompt for Gemini")

class GlobalImagePlan(BaseModel):
    md_with_placeholders: str
    images: List[ImageSpec] = Field(default_factory=list)

class State(TypedDict):
    topic: str
    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[dict]
    plan: Optional[dict] # Stored as dict to allow Pydantic v1 serialization
    
    # workers
    sections: Annotated[List[tuple], operator.add] 
    
    # reducer / images
    merged_md: str
    md_with_placeholders: str
    image_specs: List[dict]
    final: str

# ==========================================
# 2. LLM SETUP
# ==========================================
llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud", # Matches your model name
    temperature=0,
)

# ==========================================
# 3. ROUTER NODE
# ==========================================
ROUTER_SYSTEM = """You are a routing module. Decide if web research is needed.
Return STRICT JSON (no markdown):
{
  "needs_research": boolean,
  "mode": "hybrid", 
  "queries": ["query1", "query2"]
}
Modes:
- closed_book: Concepts only.
- hybrid: Concepts + Examples (Requires Research).
- open_book: News/Trends (Requires Research).
"""

def router_node(state: State) -> dict:
    print(f"--- Router Node (Topic: {state['topic']}) ---")
    response = llm.invoke(
        [
            SystemMessage(content=ROUTER_SYSTEM),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        decision = RouterDecision(**data)
        
        print(f"Mode: {decision.mode} | Research Required: {decision.needs_research}")
        return {
            "needs_research": decision.needs_research,
            "mode": decision.mode,
            "queries": decision.queries,
        }
    except Exception as e:
        print(f"Router Error: {e}. Defaulting to Hybrid.")
        return {
            "needs_research": True, 
            "mode": "hybrid", 
            "queries": [f"{state['topic']} trends 2025"]
        }

def route_next(state: State) -> str:
    return "research" if state["needs_research"] else "orchestrator"

# ==========================================
# 4. RESEARCH NODE
# ==========================================
def _tavily_search(query: str, max_results: int = 3) -> List[dict]:
    print(f"  > Searching Tavily: {query}")
    try:
        tool = TavilySearchResults(max_results=max_results)
        results = tool.invoke({"query": query})
        normalized = []
        for r in results or []:
            normalized.append({
                "title": r.get("title", "No Title"),
                "url": r.get("url", ""),
                "snippet": r.get("content", "") or r.get("snippet", ""),
                "published_at": r.get("published_date")
            })
        return normalized
    except Exception as e:
        print(f"    Tavily Error: {e}")
        return []

RESEARCH_SYSTEM = """Synthesize search results into a JSON object.
Return STRICT JSON (no markdown):
{
  "evidence": [
    { "title": "...", "url": "...", "snippet": "..." }
  ]
}
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", [])[:3]
    raw_results = []
    
    for q in queries:
        raw_results.extend(_tavily_search(q))

    if not raw_results:
        return {"evidence": []}

    print("  > Synthesizing evidence...")
    response = llm.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw Results:\n{str(raw_results)[:10000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        items = data if isinstance(data, list) else data.get("evidence", [])
        
        # Deduplicate
        valid = []
        seen = set()
        for item in items:
            if item.get("url") and item["url"] not in seen:
                valid.append(item)
                seen.add(item["url"])
                
        print(f"  > Found {len(valid)} valid evidence items.")
        return {"evidence": valid}
    except Exception as e:
        print(f"  > Research Parse Error: {e}")
        return {"evidence": []}

# ==========================================
# 5. ORCHESTRATOR NODE
# ==========================================
ORCH_SYSTEM = """Create a blog plan.
Return STRICT JSON (no markdown):
{
  "blog_title": "...",
  "audience": "...",
  "tone": "...",
  "tasks": [
    { "title": "...", "goal": "...", "bullets": ["..."], "target_words": 200 }
  ]
}
"""

def orchestrator_node(state: State) -> dict:
    print("--- Orchestrator Node ---")
    evidence = state.get("evidence", [])
    
    response = llm.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Evidence: {[e.get('title') for e in evidence[:5]]}"
                )
            ),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        plan = Plan(**data)
        # Use .dict() for Pydantic v1 compatibility
        return {"plan": plan.dict()}
    except Exception as e:
        print(f"Plan Parse Error: {e}")
        # Fallback
        fallback = Plan(
            blog_title=state["topic"], audience="Devs", tone="Tech", 
            tasks=[Task(id=1, title="Intro", goal="Explain", bullets=["Point 1"], target_words=200)]
        )
        return {"plan": fallback.dict()}

# ==========================================
# 6. FANOUT & WORKER
# ==========================================
def fanout(state: State):
    tasks_with_ids = []
    if state["plan"]:
        # plan is stored as dict in state
        plan_data = state["plan"] 
        tasks = plan_data.get("tasks", [])
        
        for i, task_data in enumerate(tasks):
            if "id" not in task_data or task_data["id"] == 0:
                task_data["id"] = i + 1
            tasks_with_ids.append(task_data)

    return [
        Send(
            "worker",
            {
                "task": task,
                "topic": state["topic"],
                "plan": state["plan"],
                "evidence": state.get("evidence", []),
            },
        )
        for task in tasks_with_ids
    ]

WORKER_SYSTEM = """You are a technical writer. Write ONE section in Markdown.
Constraints:
- Cover all bullets.
- Use evidence URLs for citations.
- Start with '## Title'.
- Do NOT output JSON. Output raw Markdown.
"""

def worker_node(payload: dict) -> dict:
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = payload.get("evidence", [])
    
    bullets_text = "\n- " + "\n- ".join(task.bullets)
    evidence_text = "\n".join([f"- {e.get('title')} ({e.get('url')})" for e in evidence[:10]])

    print(f"Writing Section: {task.title}")
    
    response = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Section: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Bullets:{bullets_text}\n"
                    f"Evidence:\n{evidence_text}\n"
                )
            ),
        ]
    )
    return {"sections": [(task.id, response.content.strip())]}

# ============================================================
# 7. REDUCER SUBGRAPH (Merge -> Image Decision -> Generation)
# ============================================================
def merge_content(state: State) -> dict:
    plan = state["plan"]
    # Sort sections by ID
    ordered = [text for _, text in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered).strip()
    title = plan.get("blog_title") if plan else "Blog Post"
    merged_md = f"# {title}\n\n{body}\n"
    return {"merged_md": merged_md}

DECIDE_IMAGES_SYSTEM = """You are an editor.
Decide if images are needed.
Rules:
- Max 3 images.
- Insert placeholders [[IMAGE_1]], [[IMAGE_2]] in the text where they belong.
- Return JSON:
{
  "md_with_placeholders": "...",
  "images": [
    { "placeholder": "[[IMAGE_1]]", "filename": "chart.png", "alt": "...", "caption": "...", "prompt": "..." }
  ]
}
"""

def decide_images(state: State) -> dict:
    print("--- Deciding Images ---")
    merged_md = state["merged_md"]
    
    # We use manual JSON parsing for robustness with DeepSeek
    response = llm.invoke(
        [
            SystemMessage(content=DECIDE_IMAGES_SYSTEM),
            HumanMessage(content=f"Text to review:\n{merged_md[:15000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        image_plan = GlobalImagePlan(**data)
        
        # Use .dict() for Pydantic v1 compatibility
        return {
            "md_with_placeholders": image_plan.md_with_placeholders,
            "image_specs": [img.dict() for img in image_plan.images],
        }
    except Exception as e:
        print(f"Image Decision Failed: {e}. Proceeding without images.")
        return {
            "md_with_placeholders": merged_md,
            "image_specs": []
        }

# ==========================================
# UPDATED: Robust Image Generation
# ==========================================
import base64

def _generate_imagen_image(prompt: str) -> bytes:
    """
    Robust generation: Tries Imagen 3, falls back to Gemini 2.0 Flash.
    """
    api_key = os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("GOOGLE_API_KEY is not set.")
    
    genai.configure(api_key=api_key)

    # 1. Try Imagen 3 (Best Quality)
    print(f"    > Attempting generation with Imagen 3: {prompt[:40]}...")
    try:
        model = genai.ImageGenerationModel("imagen-3.0-generate-001")
        response = model.generate_images(
            prompt=prompt,
            number_of_images=1,
        )
        return response.images[0].bytes
    except Exception as e_imagen:
        print(f"      [X] Imagen 3 Failed: {e_imagen}")

    # 2. Fallback: Gemini 2.0 Flash (Experimental)
    # Note: This uses the generate_content API which sometimes supports image output
    print(f"    > Attempting fallback with Gemini 2.0 Flash...")
    try:
        model = genai.GenerativeModel("gemini-2.0-flash")
        response = model.generate_content(
            f"Generate an image of: {prompt}",
            generation_config=genai.types.GenerationConfig(
                response_modalities=["IMAGE"]
            )
        )
        
        # Extract image bytes from response parts
        if response.parts:
            for part in response.parts:
                if hasattr(part, "inline_data") and part.inline_data:
                    return part.inline_data.data
        
        raise ValueError("No image data in Gemini 2.0 response")
        
    except Exception as e_flash:
        print(f"      [X] Gemini 2.0 Flash Failed: {e_flash}")
        
    # If both fail, raise the original error to trigger the placeholder text
    raise RuntimeError("All image generation models failed.")

def generate_and_place_images(state: State) -> dict:
    image_specs = state.get("image_specs", [])
    md = state.get("md_with_placeholders") or state.get("merged_md")

    if not image_specs:
        return {"final": md}

    print(f"--- Generating {len(image_specs)} Images ---")
    images_dir = Path("images")
    images_dir.mkdir(exist_ok=True)

    for spec in image_specs:
        placeholder = spec["placeholder"]
        filename = spec["filename"]
        out_path = images_dir / filename
        
        if not out_path.exists():
            try:
                img_bytes = _generate_imagen_image(spec["prompt"])
                out_path.write_bytes(img_bytes)
                print(f"    > SUCCESS: Saved {filename}")
                
                # Link the image
                img_md = f"![{spec['alt']}](images/{filename})\n*{spec['caption']}*"
                md = md.replace(placeholder, img_md)
                
            except Exception as e:
                print(f"    > FAILURE: Could not generate {filename}")
                # Fallback: Create a visible error placeholder in the blog
                error_block = (
                    f"\n> **[Image Missing]**\n"
                    f"> *Prompt: {spec.get('prompt')}*\n"
                    f"> *Error: API requests failed. Check console logs.*\n"
                )
                md = md.replace(placeholder, error_block)
        else:
            # File already exists
            print(f"    > Skipping {filename} (already exists)")
            img_md = f"![{spec['alt']}](images/{filename})\n*{spec['caption']}*"
            md = md.replace(placeholder, img_md)

    return {"final": md}

# --- Reducer Subgraph Wiring ---
reducer_graph = StateGraph(State)
reducer_graph.add_node("merge_content", merge_content)
reducer_graph.add_node("decide_images", decide_images)
reducer_graph.add_node("generate_and_place_images", generate_and_place_images)

reducer_graph.add_edge(START, "merge_content")
reducer_graph.add_edge("merge_content", "decide_images")
reducer_graph.add_edge("decide_images", "generate_and_place_images")
reducer_graph.add_edge("generate_and_place_images", END)

reducer_subgraph = reducer_graph.compile()

# ==========================================
# 8. MAIN GRAPH WIRING
# ==========================================
g = StateGraph(State)

g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_subgraph)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")

# Define exit from reducer
def save_final(state: State):
    final_md = state.get("final", "")
    plan = state.get("plan", {})
    title = plan.get("blog_title", "blog_post")
    
    # Save file
    safe_title = re.sub(r"[^a-zA-Z0-9]", "_", title)
    filename = f"{safe_title}.md"
    try:
        Path(filename).write_text(final_md, encoding="utf-8")
        print(f"SUCCESS: Saved blog to {filename}")
    except Exception as e:
        print(f"Error saving file: {e}")

g.add_node("saver", save_final)
g.add_edge("reducer", "saver")
g.add_edge("saver", END)

app = g.compile()

# ==========================================
# 9. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting DeepSeek + Gemini Blog Generator...")
    
    # Example input
    initial_state = {
        "topic": "Microservices vs Monoliths in 2025", 
        "sections": [],
        "evidence": [],
        "image_specs": [] # Initialize empty
    }
    
    try:
        app.invoke(initial_state)
        print("Done.")
    except Exception as e:
        print(f"Fatal Execution Error: {e}")

Starting DeepSeek + Gemini Blog Generator...
--- Router Node (Topic: Microservices vs Monoliths in 2025) ---
Mode: hybrid | Research Required: True
  > Searching Tavily: 2025 trends microservices vs monolith architecture
  > Searching Tavily: current adoption rates microservices monoliths 2025
  > Searching Tavily: 2025 best practices microservices monolith comparison
  > Synthesizing evidence...


KeyboardInterrupt: 

In [10]:
%pip install google-generativeai

from __future__ import annotations

import json
import re
import os
import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated, Any

# --- Pydantic V1 for LangChain Compatibility ---
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# --- LangChain & Ollama ---
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

# --- Google GenAI ---
import google.generativeai as genai

# ==========================================
# 0. HELPER: Clean JSON from Markdown
# ==========================================
def clean_json_output(text: str) -> str:
    """Removes markdown code fences to ensure json.loads can parse the output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

# ==========================================
# 1. ROBUST SCHEMAS (Pydantic V1)
# ==========================================

class Task(BaseModel):
    id: int = Field(default=0)
    title: str = Field(default="Section")

    @root_validator(pre=True)
    def map_deepseek_task_keys(cls, values: dict[str, Any]) -> dict[str, Any]:
        mapping = {
            "section_title": "title", "section_goal": "goal",
            "content": "bullets", "target_word_count": "target_words"
        }
        for old_key, new_key in mapping.items():
            if old_key in values:
                values[new_key] = values.pop(old_key)
        return values

    goal: str = Field(..., description="Section goal")
    bullets: List[str] = Field(..., min_length=1)
    target_words: int = Field(..., description="Word count")
    
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False

class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task]

    @root_validator(pre=True)
    def fix_deepseek_plan(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "blog_plan" in values: return values["blog_plan"]
        if "plan" in values and isinstance(values["plan"], list):
            return {
                "blog_title": "DeepSeek Generated Blog",
                "audience": "Developers",
                "tone": "Technical",
                "tasks": values["plan"]
            }
        return values

class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

    @root_validator(pre=True)
    def fix_deepseek_router(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "router_decision" in values:
            values = values["router_decision"]
            
        if "mode" not in values:
            values["mode"] = "hybrid" if values.get("needs_research") else "closed_book"

        if values.get("mode") in ["hybrid", "open_book"]:
            values["needs_research"] = True
            if not values.get("queries"):
                values["queries"] = ["latest trends and best practices"]

        return values

class ImageSpec(BaseModel):
    placeholder: str = Field(..., description="e.g. [[IMAGE_1]]")
    filename: str = Field(..., description="e.g. diagram.png")
    alt: str
    caption: str
    prompt: str = Field(..., description="Detailed prompt for Imagen")

class GlobalImagePlan(BaseModel):
    md_with_placeholders: str
    images: List[ImageSpec] = Field(default_factory=list)

class State(TypedDict):
    topic: str
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[dict]
    plan: Optional[dict]
    sections: Annotated[List[tuple], operator.add] 
    merged_md: str
    md_with_placeholders: str
    image_specs: List[dict]
    final: str

# ==========================================
# 2. LLM SETUP
# ==========================================
llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud", 
    temperature=0,
)

# ==========================================
# 3. ROUTER NODE
# ==========================================
ROUTER_SYSTEM = """You are a routing module. Decide if web research is needed.
Return STRICT JSON (no markdown):
{
  "needs_research": boolean,
  "mode": "hybrid", 
  "queries": ["query1", "query2"]
}
"""

def router_node(state: State) -> dict:
    print(f"--- Router Node (Topic: {state['topic']}) ---")
    response = llm.invoke(
        [
            SystemMessage(content=ROUTER_SYSTEM),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        decision = RouterDecision(**data)
        
        print(f"Mode: {decision.mode} | Research Required: {decision.needs_research}")
        return {
            "needs_research": decision.needs_research,
            "mode": decision.mode,
            "queries": decision.queries,
        }
    except Exception as e:
        print(f"Router Error: {e}. Defaulting to Hybrid.")
        return {
            "needs_research": True, "mode": "hybrid", 
            "queries": [f"{state['topic']} trends 2025"]
        }

def route_next(state: State) -> str:
    return "research" if state["needs_research"] else "orchestrator"

# ==========================================
# 4. RESEARCH NODE
# ==========================================
def _tavily_search(query: str, max_results: int = 3) -> List[dict]:
    print(f"  > Searching Tavily: {query}")
    try:
        tool = TavilySearchResults(max_results=max_results)
        results = tool.invoke({"query": query})
        normalized = []
        for r in results or []:
            normalized.append({
                "title": r.get("title", "No Title"),
                "url": r.get("url", ""),
                "snippet": r.get("content", "") or r.get("snippet", ""),
                "published_at": r.get("published_date")
            })
        return normalized
    except Exception as e:
        print(f"    Tavily Error: {e}")
        return []

RESEARCH_SYSTEM = """Synthesize search results into a JSON object.
Return STRICT JSON (no markdown):
{
  "evidence": [
    { "title": "...", "url": "...", "snippet": "..." }
  ]
}
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", [])[:3]
    raw_results = []
    
    for q in queries:
        raw_results.extend(_tavily_search(q))

    if not raw_results:
        return {"evidence": []}

    print("  > Synthesizing evidence...")
    response = llm.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw Results:\n{str(raw_results)[:10000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        items = data if isinstance(data, list) else data.get("evidence", [])
        
        valid = []
        seen = set()
        for item in items:
            if item.get("url") and item["url"] not in seen:
                valid.append(item)
                seen.add(item["url"])
                
        print(f"  > Found {len(valid)} valid evidence items.")
        return {"evidence": valid}
    except Exception as e:
        print(f"  > Research Parse Error: {e}")
        return {"evidence": []}

# ==========================================
# 5. ORCHESTRATOR NODE
# ==========================================
ORCH_SYSTEM = """Create a blog plan.
Return STRICT JSON (no markdown):
{
  "blog_title": "...",
  "audience": "...",
  "tone": "...",
  "tasks": [
    { "title": "...", "goal": "...", "bullets": ["..."], "target_words": 200 }
  ]
}
"""

def orchestrator_node(state: State) -> dict:
    print("--- Orchestrator Node ---")
    evidence = state.get("evidence", [])
    
    response = llm.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Evidence: {[e.get('title') for e in evidence[:5]]}"
                )
            ),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        plan = Plan(**data)
        return {"plan": plan.dict()}
    except Exception as e:
        print(f"Plan Parse Error: {e}")
        fallback = Plan(
            blog_title=state["topic"], audience="Devs", tone="Tech", 
            tasks=[Task(id=1, title="Intro", goal="Explain", bullets=["Point 1"], target_words=200)]
        )
        return {"plan": fallback.dict()}

# ==========================================
# 6. FANOUT & WORKER
# ==========================================
def fanout(state: State):
    tasks_with_ids = []
    if state["plan"]:
        plan_data = state["plan"] 
        tasks = plan_data.get("tasks", [])
        
        for i, task_data in enumerate(tasks):
            if "id" not in task_data or task_data["id"] == 0:
                task_data["id"] = i + 1
            tasks_with_ids.append(task_data)

    return [
        Send(
            "worker",
            {
                "task": task,
                "topic": state["topic"],
                "plan": state["plan"],
                "evidence": state.get("evidence", []),
            },
        )
        for task in tasks_with_ids
    ]

WORKER_SYSTEM = """You are a technical writer. Write ONE section in Markdown.
Constraints:
- Cover all bullets.
- Use evidence URLs for citations.
- Start with '## Title'.
- Do NOT output JSON. Output raw Markdown.
"""

def worker_node(payload: dict) -> dict:
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = payload.get("evidence", [])
    
    bullets_text = "\n- " + "\n- ".join(task.bullets)
    evidence_text = "\n".join([f"- {e.get('title')} ({e.get('url')})" for e in evidence[:10]])

    print(f"Writing Section: {task.title}")
    
    response = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Section: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Bullets:{bullets_text}\n"
                    f"Evidence:\n{evidence_text}\n"
                )
            ),
        ]
    )
    return {"sections": [(task.id, response.content.strip())]}

# ============================================================
# 7. REDUCER SUBGRAPH (Merge -> Image Decision -> Generation)
# ============================================================
def merge_content(state: State) -> dict:
    plan = state["plan"]
    ordered = [text for _, text in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered).strip()
    title = plan.get("blog_title") if plan else "Blog Post"
    merged_md = f"# {title}\n\n{body}\n"
    return {"merged_md": merged_md}

DECIDE_IMAGES_SYSTEM = """You are an editor.
Decide if images are needed.
Rules:
- Max 3 images.
- Insert placeholders [[IMAGE_1]], [[IMAGE_2]] in the text where they belong.
- Return JSON:
{
  "md_with_placeholders": "...",
  "images": [
    { "placeholder": "[[IMAGE_1]]", "filename": "chart.png", "alt": "...", "caption": "...", "prompt": "..." }
  ]
}
"""

def decide_images(state: State) -> dict:
    print("--- Deciding Images ---")
    merged_md = state["merged_md"]
    
    response = llm.invoke(
        [
            SystemMessage(content=DECIDE_IMAGES_SYSTEM),
            HumanMessage(content=f"Text to review:\n{merged_md[:15000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        image_plan = GlobalImagePlan(**data)
        
        return {
            "md_with_placeholders": image_plan.md_with_placeholders,
            "image_specs": [img.dict() for img in image_plan.images],
        }
    except Exception as e:
        print(f"Image Decision Failed: {e}. Proceeding without images.")
        return {
            "md_with_placeholders": merged_md,
            "image_specs": []
        }

# ==========================================
# UPDATED: Robust Image Generation
# ==========================================
import base64

# ==========================================
# NEW: Free Image Generation (No API Key Needed)
# ==========================================
import requests
import time

def _generate_image_via_pollinations(prompt: str, filename: str) -> bytes:
    """
    Generates an image using the free Pollinations.ai API.
    No API Key required.
    """
    # 1. Clean the prompt for the URL (spaces to %20)
    # We keep it simple to ensure the API accepts it
    safe_prompt = prompt[:100].replace(" ", "%20") 
    
    # 2. Construct URL (seed ensures reproducibility)
    # nologo=true removes the watermark
    url = f"https://image.pollinations.ai/prompt/{safe_prompt}?nologo=true"
    
    print(f"    > Requesting image: {url[:60]}...")
    
    # 3. Fetch image
    response = requests.get(url, timeout=30)
    if response.status_code == 200:
        return response.content
    else:
        raise ValueError(f"Pollinations API failed with status {response.status_code}")

def generate_and_place_images(state: State) -> dict:
    image_specs = state.get("image_specs", [])
    md = state.get("md_with_placeholders") or state.get("merged_md")

    if not image_specs:
        return {"final": md}

    print(f"--- Generating {len(image_specs)} Images (via Pollinations) ---")
    
    # Create images directory
    images_dir = Path("images")
    images_dir.mkdir(exist_ok=True)

    for spec in image_specs:
        placeholder = spec["placeholder"]
        filename = spec["filename"]
        out_path = images_dir / filename
        
        if not out_path.exists():
            try:
                # Use the new Pollinations function
                img_bytes = _generate_image_via_pollinations(spec["prompt"], filename)
                out_path.write_bytes(img_bytes)
                print(f"    > SUCCESS: Saved {filename}")
                
                # Replace placeholder with Markdown image link
                img_md = f"![{spec['alt']}](images/{filename})\n*{spec['caption']}*"
                md = md.replace(placeholder, img_md)
                
                # Sleep briefly to be nice to the free API
                time.sleep(1) 
                
            except Exception as e:
                print(f"    > FAILURE: Could not generate {filename}. Error: {e}")
                error_block = f"> *[Image Generation Failed: {spec.get('caption')}]*"
                md = md.replace(placeholder, error_block)
        else:
            print(f"    > Skipping {filename} (already exists)")
            img_md = f"![{spec['alt']}](images/{filename})\n*{spec['caption']}*"
            md = md.replace(placeholder, img_md)

    return {"final": md}

# --- Reducer Subgraph Wiring ---
reducer_graph = StateGraph(State)
reducer_graph.add_node("merge_content", merge_content)
reducer_graph.add_node("decide_images", decide_images)
reducer_graph.add_node("generate_and_place_images", generate_and_place_images)

reducer_graph.add_edge(START, "merge_content")
reducer_graph.add_edge("merge_content", "decide_images")
reducer_graph.add_edge("decide_images", "generate_and_place_images")
reducer_graph.add_edge("generate_and_place_images", END)

reducer_subgraph = reducer_graph.compile()

# ==========================================
# 8. MAIN GRAPH WIRING
# ==========================================
g = StateGraph(State)

g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_subgraph)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")

# Define exit from reducer
def save_final(state: State):
    final_md = state.get("final", "")
    plan = state.get("plan", {})
    title = plan.get("blog_title", "blog_post")
    
    safe_title = re.sub(r"[^a-zA-Z0-9]", "_", title)
    filename = f"{safe_title}.md"
    try:
        Path(filename).write_text(final_md, encoding="utf-8")
        print(f"SUCCESS: Saved blog to {filename}")
    except Exception as e:
        print(f"Error saving file: {e}")

g.add_node("saver", save_final)
g.add_edge("reducer", "saver")
g.add_edge("saver", END)

app = g.compile()

# ==========================================
# 9. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting DeepSeek + Gemini Blog Generator...")
    
    initial_state = {
        "topic": "Self-Attention in Transformer Architecture", 
        "sections": [],
        "evidence": [],
        "image_specs": [] 
    }
    
    try:
        app.invoke(initial_state)
        print("Done.")
    except Exception as e:
        print(f"Fatal Execution Error: {e}")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Starting DeepSeek + Gemini Blog Generator...
--- Router Node (Topic: Self-Attention in Transformer Architecture) ---
Mode: hybrid | Research Required: True
  > Searching Tavily: latest trends and best practices
  > Synthesizing evidence...
  > Found 3 valid evidence items.
--- Orchestrator Node ---
Writing Section: Beyond the Hype: Making Sense of Self-Attention for Non-Tech Leaders
Writing Section: Self-Attention Meets Business Strategy: Three 2026 Applications
Writing Section: Your 2026 Roadmap: Preparing for the Self-Attention Revolution
Writing Section: The Competitive Edge: Why Early Adoption Matters
--- Deciding Images ---
--- Generating 3 Images (via Pollinations) ---
    > Requesting image: https://image.pollinations.ai/prompt/Create%20a%20clean,%20p...
    > SUCCESS: Saved self-attention-analogy.png
    > Requesting image: https://image.pollinations.ai/prompt/Create%20a%20comparison...
    > SUCCESS: Saved planning-roadmap-timeline.png
    > Requesting image: https://image.pol

In [12]:
%pip install google-generativeai langchain-ollama langgraph pydantic tavily-python

from __future__ import annotations

import json
import re
import os
import operator
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated, Any

# --- Pydantic V1 for LangChain Compatibility ---
from pydantic.v1 import BaseModel, Field, root_validator
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

# --- LangChain & Ollama ---
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

# ==========================================
# 0. HELPER: Clean JSON from Markdown
# ==========================================
def clean_json_output(text: str) -> str:
    """Removes markdown code fences to ensure json.loads can parse the output."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

# ==========================================
# 1. ROBUST SCHEMAS (Pydantic V1)
# ==========================================

class Task(BaseModel):
    id: int = Field(default=0)
    title: str = Field(default="Section")
    goal: str = Field(..., description="Section goal")
    bullets: List[str] = Field(..., min_length=1)
    target_words: int = Field(..., description="Word count")
    
    @root_validator(pre=True)
    def map_deepseek_task_keys(cls, values: dict[str, Any]) -> dict[str, Any]:
        mapping = {
            "section_title": "title", "section_goal": "goal",
            "content": "bullets", "target_word_count": "target_words"
        }
        for old_key, new_key in mapping.items():
            if old_key in values:
                values[new_key] = values.pop(old_key)
        return values

class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    tasks: List[Task]

    @root_validator(pre=True)
    def fix_deepseek_plan(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "blog_plan" in values: return values["blog_plan"]
        if "plan" in values and isinstance(values["plan"], list):
            return {
                "blog_title": "DeepSeek Generated Blog",
                "audience": "Developers",
                "tone": "Technical",
                "tasks": values["plan"]
            }
        return values

class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

    @root_validator(pre=True)
    def fix_deepseek_router(cls, values: dict[str, Any]) -> dict[str, Any]:
        if "router_decision" in values:
            values = values["router_decision"]
            
        if "mode" not in values:
            values["mode"] = "hybrid" if values.get("needs_research") else "closed_book"

        if values.get("mode") in ["hybrid", "open_book"]:
            values["needs_research"] = True
            if not values.get("queries"):
                values["queries"] = ["latest trends and best practices"]
        return values

# --- Diagram Schemas ---
class DiagramSpec(BaseModel):
    placeholder: str = Field(..., description="e.g. [[DIAGRAM_1]]")
    type: Literal["flowchart", "sequence", "class", "state", "er"]
    code: str = Field(..., description="Raw Mermaid code without markdown backticks")
    caption: str

class GlobalDiagramPlan(BaseModel):
    md_with_placeholders: str
    diagrams: List[DiagramSpec] = Field(default_factory=list)

class State(TypedDict):
    topic: str
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[dict]
    plan: Optional[dict]
    sections: Annotated[List[tuple], operator.add] 
    merged_md: str
    md_with_placeholders: str
    diagram_specs: List[dict]
    final: str

# ==========================================
# 2. LLM SETUP
# ==========================================
llm = ChatOllama(
    model="deepseek-v3.1:671b-cloud", 
    temperature=0,
)

# ==========================================
# 3. ROUTER NODE
# ==========================================
ROUTER_SYSTEM = """You are a routing module. Decide if web research is needed.
Return STRICT JSON (no markdown):
{
  "needs_research": boolean,
  "mode": "hybrid", 
  "queries": ["query1", "query2"]
}
"""

def router_node(state: State) -> dict:
    print(f"--- Router Node (Topic: {state['topic']}) ---")
    response = llm.invoke(
        [
            SystemMessage(content=ROUTER_SYSTEM),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        decision = RouterDecision(**data)
        print(f"Mode: {decision.mode} | Research Required: {decision.needs_research}")
        return {
            "needs_research": decision.needs_research,
            "mode": decision.mode,
            "queries": decision.queries,
        }
    except Exception as e:
        print(f"Router Error: {e}. Defaulting to Hybrid.")
        return {"needs_research": True, "mode": "hybrid", "queries": [f"{state['topic']} trends"]}

def route_next(state: State) -> str:
    return "research" if state["needs_research"] else "orchestrator"

# ==========================================
# 4. RESEARCH NODE
# ==========================================
def _tavily_search(query: str, max_results: int = 3) -> List[dict]:
    print(f"  > Searching Tavily: {query}")
    try:
        tool = TavilySearchResults(max_results=max_results)
        results = tool.invoke({"query": query})
        normalized = []
        for r in results or []:
            normalized.append({
                "title": r.get("title", "No Title"),
                "url": r.get("url", ""),
                "snippet": r.get("content", "") or r.get("snippet", ""),
            })
        return normalized
    except Exception as e:
        print(f"    Tavily Error: {e}")
        return []

RESEARCH_SYSTEM = """Synthesize search results into a JSON object.
Return STRICT JSON (no markdown):
{
  "evidence": [
    { "title": "...", "url": "...", "snippet": "..." }
  ]
}
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", [])[:3]
    raw_results = []
    
    for q in queries:
        raw_results.extend(_tavily_search(q))

    if not raw_results:
        return {"evidence": []}

    print("  > Synthesizing evidence...")
    response = llm.invoke(
        [
            SystemMessage(content=RESEARCH_SYSTEM),
            HumanMessage(content=f"Raw Results:\n{str(raw_results)[:10000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        items = data if isinstance(data, list) else data.get("evidence", [])
        return {"evidence": items}
    except Exception as e:
        print(f"  > Research Parse Error: {e}")
        return {"evidence": []}

# ==========================================
# 5. ORCHESTRATOR NODE
# ==========================================
ORCH_SYSTEM = """Create a blog plan.
Return STRICT JSON (no markdown):
{
  "blog_title": "...",
  "audience": "...",
  "tone": "...",
  "tasks": [
    { "title": "...", "goal": "...", "bullets": ["..."], "target_words": 200 }
  ]
}
"""

def orchestrator_node(state: State) -> dict:
    print("--- Orchestrator Node ---")
    evidence = state.get("evidence", [])
    
    response = llm.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Evidence: {[e.get('title') for e in evidence[:5]]}"
                )
            ),
        ]
    )
    
    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        plan = Plan(**data)
        return {"plan": plan.dict()}
    except Exception as e:
        print(f"Plan Parse Error: {e}")
        fallback = Plan(
            blog_title=state["topic"], audience="Devs", tone="Tech", 
            tasks=[Task(id=1, title="Intro", goal="Explain", bullets=["Point 1"], target_words=200)]
        )
        return {"plan": fallback.dict()}

# ==========================================
# 6. FANOUT & WORKER
# ==========================================
def fanout(state: State):
    tasks_with_ids = []
    if state["plan"]:
        plan_data = state["plan"] 
        tasks = plan_data.get("tasks", [])
        for i, task_data in enumerate(tasks):
            if "id" not in task_data or task_data["id"] == 0:
                task_data["id"] = i + 1
            tasks_with_ids.append(task_data)

    return [
        Send(
            "worker",
            {
                "task": task,
                "topic": state["topic"],
                "plan": state["plan"],
                "evidence": state.get("evidence", []),
            },
        )
        for task in tasks_with_ids
    ]

# --- UPDATED: Worker System Prompt (Encourages Tables) ---
WORKER_SYSTEM = """You are a technical writer. Write ONE section in Markdown.
Constraints:
- Cover all bullets.
- Use evidence URLs for citations.
- Start with '## Title'.
- Use Markdown tables for comparisons or data if appropriate.
- Do NOT output JSON. Output raw Markdown.
"""

def worker_node(payload: dict) -> dict:
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = payload.get("evidence", [])
    
    bullets_text = "\n- " + "\n- ".join(task.bullets)
    evidence_text = "\n".join([f"- {e.get('title')} ({e.get('url')})" for e in evidence[:10]])

    print(f"Writing Section: {task.title}")
    
    response = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Section: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Bullets:{bullets_text}\n"
                    f"Evidence:\n{evidence_text}\n"
                )
            ),
        ]
    )
    return {"sections": [(task.id, response.content.strip())]}

# ============================================================
# 7. REDUCER SUBGRAPH
# ============================================================
def merge_content(state: State) -> dict:
    plan = state["plan"]
    ordered = [text for _, text in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered).strip()
    title = plan.get("blog_title") if plan else "Blog Post"
    merged_md = f"# {title}\n\n{body}\n"
    return {"merged_md": merged_md}

# --- UPDATED: Diagram Decision Prompt (Strict Max 2) ---
DECIDE_DIAGRAMS_SYSTEM = """You are a Technical Editor.
Identify complex concepts that need visualization (logic flows, system architecture).

Rules:
1. STRICT LIMIT: Maximum 2 diagrams total.
2. Use placeholders [[DIAGRAM_1]] in the text where they fit best.
3. Return STRICT JSON:
{
  "md_with_placeholders": "...",
  "diagrams": [
    { 
      "placeholder": "[[DIAGRAM_1]]", 
      "type": "flowchart", 
      "code": "graph TD; A[Start]-->B[End];", 
      "caption": "Figure 1: Process Flow" 
    }
  ]
}
Note: 'code' must be raw mermaid syntax. Do not wrap 'code' in markdown backticks in JSON.
"""

def decide_diagrams(state: State) -> dict:
    print("--- Deciding Diagrams (Mermaid) ---")
    merged_md = state["merged_md"]
    
    response = llm.invoke(
        [
            SystemMessage(content=DECIDE_DIAGRAMS_SYSTEM),
            HumanMessage(content=f"Text to review:\n{merged_md[:15000]}"),
        ]
    )

    try:
        clean_text = clean_json_output(response.content)
        data = json.loads(clean_text)
        diagram_plan = GlobalDiagramPlan(**data)
        
        # Enforce limit in code just in case LLM hallucinations
        final_diagrams = diagram_plan.diagrams[:2]
        
        print(f"  > Generated {len(final_diagrams)} diagrams.")
        return {
            "md_with_placeholders": diagram_plan.md_with_placeholders,
            "diagram_specs": [d.dict() for d in final_diagrams],
        }
    except Exception as e:
        print(f"Diagram Decision Failed: {e}. Proceeding without diagrams.")
        return {
            "md_with_placeholders": merged_md,
            "diagram_specs": []
        }

def inject_diagrams(state: State) -> dict:
    specs = state.get("diagram_specs", [])
    md = state.get("md_with_placeholders") or state.get("merged_md")

    if not specs:
        return {"final": md}

    print("--- Injecting Diagrams into Markdown ---")
    
    for spec in specs:
        placeholder = spec["placeholder"]
        code = spec["code"]
        caption = spec["caption"]
        
        # Format as a standard Mermaid code block for Markdown
        mermaid_block = (
            f"\n\n#### {caption}\n"
            f"```mermaid\n"
            f"{code}\n"
            f"```\n"
        )
        
        if placeholder in md:
            md = md.replace(placeholder, mermaid_block)
            print(f"  > Injected {spec['type']} at {placeholder}")
        else:
            md += mermaid_block

    return {"final": md}

# --- Reducer Subgraph Wiring ---
reducer_graph = StateGraph(State)
reducer_graph.add_node("merge_content", merge_content)
reducer_graph.add_node("decide_diagrams", decide_diagrams)
reducer_graph.add_node("inject_diagrams", inject_diagrams)

reducer_graph.add_edge(START, "merge_content")
reducer_graph.add_edge("merge_content", "decide_diagrams")
reducer_graph.add_edge("decide_diagrams", "inject_diagrams")
reducer_graph.add_edge("inject_diagrams", END)

reducer_subgraph = reducer_graph.compile()

# ==========================================
# 8. MAIN GRAPH WIRING
# ==========================================
g = StateGraph(State)

g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_subgraph)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")
g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")

# Define exit from reducer
def save_final(state: State):
    final_md = state.get("final", "")
    plan = state.get("plan", {})
    title = plan.get("blog_title", "blog_post")
    
    safe_title = re.sub(r"[^a-zA-Z0-9]", "_", title)
    filename = f"{safe_title}.md"
    try:
        Path(filename).write_text(final_md, encoding="utf-8")
        print(f"SUCCESS: Saved blog to {filename}")
        print("Note: Use a Markdown viewer with Mermaid support to view diagrams.")
    except Exception as e:
        print(f"Error saving file: {e}")

g.add_node("saver", save_final)
g.add_edge("reducer", "saver")
g.add_edge("saver", END)

app = g.compile()

# ==========================================
# 9. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting DeepSeek Blog Generator...")
    
    initial_state = {
        "topic": "Self-Attention in Transformer Architecture", 
        "sections": [],
        "evidence": [],
        "diagram_specs": [] 
    }
    
    try:
        app.invoke(initial_state)
        print("Done.")
    except Exception as e:
        print(f"Fatal Execution Error: {e}")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Starting DeepSeek Blog Generator...
--- Router Node (Topic: Self-Attention in Transformer Architecture) ---
Mode: hybrid | Research Required: True
  > Searching Tavily: latest trends and best practices
  > Synthesizing evidence...
--- Orchestrator Node ---
Writing Section: Introduction: The Engine for Next-Gen Tech
Writing Section: Demystifying Self-Attention: It's All About Relationships
Writing Section: Trend #1: Hyper-Personalized Social Media (Connecting to Hootsuite)
Writing Section: Trend #2: The Smart, Adaptive Factory (Connecting to ATS)
Writing Section: Trend #3: Enterprise AI That Understands Context (Connecting to Deloitte)
Writing Section: Conclusion: The Foundational Layer of Future Innovation
--- Deciding Diagrams (Mermaid) ---
  > Generated 2 diagrams.
--- Injecting Diagrams into Markdown ---
  > Injected flowchart at [[DIAGRAM_1]]
  > Injected flowchart at [[DIAGRAM_2]]
SUCCESS: Saved blog to Beyond_2025_